In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

from fraud_engine.data.load import DEFAULT_CONFIG_PATH, load_config

In [ ]:
# A notebook's cwd is unreliable - anchor to the repo root instead.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

config = load_config(ROOT / DEFAULT_CONFIG_PATH)

# config.yaml paths are repo-root-relative.
interim_path = ROOT / config["paths"]["interim"]

In [ ]:
# Hold back the tail: Phase 02 carves the test set from it, and a boundary chosen
# after seeing its fraud rate is contaminated.
#
# The number lives in config, not here. It is a record of what has already been
# seen rather than a preference, and Phase 02 has to honour it - VAL and TEST
# must both start after it. A constant defined in a notebook cell binds nothing.
EDA_MAX_DAY = config["eda"]["horizon_day"]
EDA_FILTER = [("day", "<=", EDA_MAX_DAY)]


# One place enforces the horizon - a bare read_parquet later would span all 182 days.
def read_eda(columns):
    return pd.read_parquet(interim_path, columns=columns, filters=EDA_FILTER)

### Section 1 - base rate and time span

How much fraud there is, and over how long — measured inside the EDA horizon, not
across the full file. The base rate fixes the class imbalance every later design
decision has to survive, and is the reason accuracy is never reported here.

In [ ]:
df = read_eda(["day", "isFraud", "TransactionAmt"])

row_count = len(df)
print(f"Row count: {row_count}")

day_span = {
    "first_day": int(df["day"].min()),
    "last_day": int(df["day"].max()),
    "distinct_day_count": df["day"].nunique(),
}
print(f"Day span: {day_span}")

fraud_count = (df["isFraud"] == 1).sum()
print(f"Fraud count: {fraud_count}")

fraud_rate = fraud_count / row_count
print(f"Fraud rate: {fraud_rate:.3%}")

In [ ]:
# The base rate is only interesting next to what it makes possible. Read the committed
# capacity rather than restating it, so a change to cost_matrix.yaml cannot leave this
# prose stale.
cost_matrix = load_config(ROOT / "config/cost_matrix.yaml")
review_capacity = cost_matrix["constraints"]["review_capacity"]["value"]

print(f"always-'not fraud' accuracy: {1 - fraud_rate:.3%}")

days = day_span["distinct_day_count"]
transactions_per_day = row_count / days
frauds_per_day = fraud_count / days
reviews_per_day = review_capacity * transactions_per_day

print(f"\n{transactions_per_day:,.0f} transactions/day, {frauds_per_day:,.1f} frauds/day")
print(f"review capacity {review_capacity:.1%} of volume = {reviews_per_day:,.1f} slots/day")
print(f"perfect-ranker recall ceiling at that capacity: {reviews_per_day / frauds_per_day:.1%}")

#### What the numbers say

414,542 transactions over 120 days (days 1–120 of the file's 182), no missing days.
14,600 of them are fraud — a base rate of **3.522%**.

Two consequences follow directly from that rate.

**Accuracy is unusable.** A model that predicts "not fraud" for every row scores
**96.478%**. Any accuracy figure quoted for this problem is describing the class balance,
not the model, which is why PR-AUC and recall@capacity are the metrics here.

**Manual review cannot be the answer on its own.** At ~3,455 transactions/day and the
`review_capacity: 0.01` committed in `cost_matrix.yaml`, the queue holds ~35 reviews/day
against ~122 frauds/day. Even a *perfect* ranker that spent every review slot on a true
fraud would top out at **28.4% recall**. The remaining ~72% has to be handled by the
allow/block decision, which is the constraint the Phase 06 cost policy exists to resolve.

`day` is an offset from an unpublished reference point, so this is a duration, not a date
range — no calendar claims can be made from it.

### Section 2 - fraud rate over time

Daily fraud rate across the horizon, with a 7-day centred rolling mean over the raw
series and transaction volume on the panel beneath — a rate move that coincides with
a volume move has a different explanation from one that does not.

The figure is saved to `reports/figures/`. Whether the
rate holds steady or drifts is the evidence the Phase 02 temporal split rests on.

In [ ]:
daily = df.groupby("day").agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
print(daily.shape)

In [ ]:
# The rate is only interpretable next to the volume that produced it.
fig, (ax_rate, ax_vol) = plt.subplots(2, 1, sharex=True, figsize=(11, 6), height_ratios=[2, 1])

# At ~3,000 transactions/day the raw series is mostly binomial noise.
ax_rate.plot(daily.index, daily["rate"], lw=1, alpha=0.35, label="daily")
ax_rate.plot(
    daily.index,
    daily["rate"].rolling(7, center=True).mean(),
    lw=2,
    label="7-day rolling mean (centred)",
)

# From zero: autoscale would amplify the noise into a trend.
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
ax_rate.set_title(f"Fraud rate over time (days 1-{EDA_MAX_DAY})")
ax_rate.legend(loc="upper left", frameon=False)
ax_rate.grid(alpha=0.25)

ax_vol.fill_between(daily.index, daily["volume"], alpha=0.4, lw=0)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
# `day` is an offset from an unpublished reference, not a calendar date.
ax_vol.set_xlabel("day (relative to an unpublished reference, not a calendar date)")
ax_vol.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# The findings below turn on whether a rate move came from the numerator or the
# denominator, so both are needed - a rate alone cannot tell them apart.
daily_parts = df.groupby("day")["isFraud"].agg(frauds="sum", volume="size")
daily_parts["legit"] = daily_parts["volume"] - daily_parts["frauds"]
daily_parts["rate"] = daily_parts["frauds"] / daily_parts["volume"]
smoothed = daily_parts.rolling(7, center=True).mean()

trough_day = int(smoothed["rate"].idxmin())
peak_volume_day = int(smoothed["volume"].idxmax())
print(f"smoothed rate troughs on day {trough_day} at {smoothed.loc[trough_day, 'rate']:.2%}")
print(f"smoothed volume peaks on day {peak_volume_day}")
print(
    f"rate/volume correlation, days 1-30 (smoothed): "
    f"{smoothed.loc[1:30, 'rate'].corr(smoothed.loc[1:30, 'volume']):+.2f}"
)

print("\n30-day window means:")
for lo, hi in [(1, 30), (31, 60), (61, 90), (91, 120)]:
    window = daily_parts.loc[lo:hi]
    print(f"  days {lo:>3}-{hi:<3} rate {window.frauds.sum() / window.volume.sum():.2%}")

print("\nnumerator vs denominator, on the smoothed series:")
for label, a, b in [("early dip", 8, 22), ("late rise", 95, 115)]:
    legit_change = smoothed.loc[b, "legit"] / smoothed.loc[a, "legit"] - 1
    fraud_change = smoothed.loc[b, "frauds"] / smoothed.loc[a, "frauds"] - 1
    print(f"  {label} (day {a} -> {b}): legit {legit_change:+.0%}, fraud count {fraud_change:+.0%}")

print(
    f"\ndays 91-100 rate {daily_parts.loc[91:100].frauds.sum() / daily_parts.loc[91:100].volume.sum():.2%}"
    f" -> days 111-120 {daily_parts.loc[111:120].frauds.sum() / daily_parts.loc[111:120].volume.sum():.2%}"
)

#### What the chart shows

The fraud rate is **not stationary** across the window.

Days 1–22 show volume climbing 55% while the daily fraud *count* stays flat (−3%), so the
rate falls to a 1.83% trough purely by dilution — the surge is legitimate customers, not
quieter attackers. Volume peaks and the rate bottoms on the same day (22); across days
1–30 the smoothed rate and volume correlate at **−0.88**.

After that the series steps to a new level rather than continuing to trend. The pooled
rates for days 31–60, 61–90 and 91–120 are 4.00%, 4.04% and 3.95% — but those flat
averages hide real movement inside them. Within the last window the rate climbs from 3.36%
(days 91–100) to 4.74% (days 111–120), driven by a 20% rise in daily fraud count against a
25% fall in legitimate volume. That rise is *not* dilution: it is more fraud against a
smaller base.

*(Rates here are pooled — frauds ÷ transactions in the window. An unweighted mean of daily
rates would over-weight low-volume days.)*

The 30-day window width is a choice, and the flatness of those means is partly an artifact
of it.

**Implication for Phase 02.** A random split would scatter both regimes across train and
test, letting the model learn from a fraud environment it would not have had in
production. The split must be temporal.

### Section 3 - hour alignment and time of day

`TransactionDT` counts seconds from a reference point Vesta never published, so `hour`
(`seconds // 3600 % 24`) is a consistent 24-hour cycle but not necessarily wall-clock time.
Bucket 0 is midnight only if that reference is midnight-aligned — `min(TransactionDT)` is
exactly 86,400, one whole day, which hints at it but is an inference, not a fact.

**The volume curve is the test.** Human commerce has a deep overnight trough. If one appears
at plausible night hours, `hour` means hour-of-day and time-of-day claims are legitimate. If
the curve is flat or oddly phased, `hour` stays a usable cyclic feature but no time-of-day
claim can be made from it — and the verdict goes back into the `add_time_columns` docstring.

In [ ]:
hourly = (
    read_eda(["hour", "isFraud"])
    .groupby("hour")
    .agg(rate=("isFraud", "mean"), volume=("isFraud", "size"))
)
print(hourly.shape)

In [ ]:
# Volume on top: this curve is the alignment test, the rate is only interpretable once it
# is settled. No smoothing - each bucket holds ~19,000 transactions (noise is +/-0.15pp),
# and rolling() would treat hours 0 and 23 as distant rather than adjacent, blanking
# exactly the midnight window being tested.
fig, (ax_vol, ax_rate) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))

ax_vol.bar(hourly.index, hourly["volume"], width=0.85, alpha=0.7)
ax_vol.set_ylim(0, None)
ax_vol.set_ylabel("transactions")
ax_vol.set_title(f"Volume and fraud rate by hour bucket (days 1-{EDA_MAX_DAY})")
ax_vol.grid(alpha=0.25)

ax_rate.plot(hourly.index, hourly["rate"], lw=2, marker="o", ms=4)
ax_rate.set_ylim(0, None)
ax_rate.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax_rate.set_ylabel("fraud rate")
# Bucket 0 is midnight only if the unpublished reference is midnight-aligned.
ax_rate.set_xlabel("hour bucket (0-23; bucket 0 is not necessarily midnight)")
ax_rate.set_xticks(range(0, 24, 2))
ax_rate.grid(alpha=0.25)

fig.tight_layout()

# savefig before show - the inline backend closes the figure.
fig.savefig(ROOT / "reports/figures/fraud_rate_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# The alignment argument rests on the size of the swing and on where the trough sits, so
# both are computed rather than read off the chart.
trough_bucket, peak_bucket = int(hourly["volume"].idxmin()), int(hourly["volume"].idxmax())
print(
    f"volume trough bucket {trough_bucket} ({hourly.loc[trough_bucket, 'volume']:,}), "
    f"peak bucket {peak_bucket} ({hourly.loc[peak_bucket, 'volume']:,}), "
    f"swing {hourly['volume'].max() / hourly['volume'].min():.1f}x"
)

# If bucket 0 were midnight M, each landmark would fall at (bucket - M) mod 24. Only
# offsets putting the trough in the pre-dawn hours describe a human day.
print("\nif midnight = bucket M, the trough and peak fall at:")
for midnight in (4, 5, 6, 8, 10, 12):
    trough_local = (trough_bucket - midnight) % 24
    peak_local = (peak_bucket - midnight) % 24
    verdict = "plausible" if 2 <= trough_local <= 6 else "NO - not a human low"
    print(f"  M={midnight:>2}: trough {trough_local:>2}:00, peak {peak_local:>2}:00   {verdict}")

# Same numerator/denominator question as section 2, by hour.
hourly_parts = (
    read_eda(["hour", "isFraud"]).groupby("hour")["isFraud"].agg(frauds="sum", volume="size")
)
hourly_parts["legit"] = hourly_parts["volume"] - hourly_parts["frauds"]
hourly_parts["rate"] = hourly_parts["frauds"] / hourly_parts["volume"]

rate_peak = int(hourly_parts["rate"].idxmax())
plateau = hourly_parts.loc[17:23]
print(
    f"\nrate peaks at bucket {rate_peak}: {hourly_parts.loc[rate_peak, 'rate']:.2%}"
    f"  vs buckets 17-23: {plateau.frauds.sum() / plateau.volume.sum():.2%}"
)
print(
    f"  bucket {rate_peak}: {hourly_parts.loc[rate_peak, 'frauds'] / EDA_MAX_DAY:.1f} frauds/day, "
    f"{hourly_parts.loc[rate_peak, 'legit'] / EDA_MAX_DAY:.0f} legit/day"
)
print(
    f"  plateau mean: {plateau.frauds.mean() / EDA_MAX_DAY:.1f} frauds/day, "
    f"{plateau.legit.mean() / EDA_MAX_DAY:.0f} legit/day"
)
print(
    f"  relative to plateau - fraud {hourly_parts.loc[rate_peak, 'frauds'] / plateau.frauds.mean():.2f}x, "
    f"legit {hourly_parts.loc[rate_peak, 'legit'] / plateau.legit.mean():.2f}x"
)

#### What the chart shows

**Bucket 0 is not midnight.** Volume swings 19× between bucket 9 (1,580) and bucket 19
(30,242) — an unmistakable diurnal cycle, but phased wrong for bucket 0 to be midnight,
which would mean the fewest transactions at 9am and the most between 7pm and 1am.

Placing the trough at a plausible pre-dawn hour puts midnight at **bucket 4–6**. Past
bucket 6 the mapping breaks: at bucket 8 the daily low lands at 1am, at bucket 12 at 9pm.
An offset of roughly +5 hours matches US Eastern's offset from UTC, which would be the
expected artifact of UTC timestamps against a mostly-US customer base — a hypothesis with
a mechanism, not a documented fact.

Consequence: `hour` is a **cyclic feature, not a wall-clock label**, and no time-of-day
claim can be made without carrying that offset as a stated assumption. The verdict is
recorded in the `add_time_columns` docstring, which previously left the question open.

#### The rate peak is a denominator effect

The fraud rate peaks at bucket 8 (10.01%, against 3.45% across buckets 17–23), and that
peak sits exactly where volume is lowest. But the counts run the other way:

| | frauds/day | legit/day | rate |
|---|---|---|---|
| bucket 8 | 1.6 | 15 | 10.01% |
| buckets 17–23 | 8.5 | 237 | 3.45% |

Relative to the plateau, fraud falls to 0.19× while legitimate activity falls to 0.06×.
Both collapse overnight; legitimate activity collapses harder. **Fraud does not peak there
— it declines more slowly than everything around it.** The hours with the most fraud by
count are the plateau hours, which carry roughly five times more fraud per day.

This is section 2's mechanism inverted: there, a surge in legitimate volume diluted a flat
fraud count; here, a collapse in legitimate volume concentrates a falling one. Rate and
count answer different questions, and only the count says where the money is.

### Section 4 - amount distribution and USD exposure

The headline result of this project is denominated in USD, so this section establishes
the denominator: how much money moves, how much of it is fraud, and whether that loss
sits in a few large transactions or spreads across many.

Two comparisons carry it — fraud against legitimate at each percentile, and fraud's
share of USD against its share of transaction count.

In [ ]:
amounts_df = read_eda(["TransactionAmt", "isFraud"])

qs = [0.25, 0.5, 0.75, 0.9, 0.975, 0.99]

amounts_df.groupby("isFraud")["TransactionAmt"].describe(percentiles=qs)

In [ ]:
all_amounts = amounts_df["TransactionAmt"]
fraud_amounts = amounts_df.loc[amounts_df["isFraud"] == 1, "TransactionAmt"]
legit_amounts = amounts_df.loc[amounts_df["isFraud"] == 0, "TransactionAmt"]

total_amount = all_amounts.sum()
total_fraud_amount = fraud_amounts.sum()

print(f"Total amount:       {total_amount:>13,.2f} USD")
print(f"Total fraud amount: {total_fraud_amount:>13,.2f} USD")

# Fraud takes a bigger bite of the money than of the count, by roughly the gap
# between the two class means.
print(f"\nfraud share of USD:   {total_fraud_amount / total_amount:.3%}")
print(f"fraud share of count: {len(fraud_amounts) / len(all_amounts):.3%}")


def top_share(values, fraction):
    """Share of total USD held by the largest `fraction` of transactions."""
    return values.nlargest(int(fraction * len(values))).sum() / values.sum()


# Concentration means nothing without a baseline - every heavy-tailed distribution
# looks concentrated. The question is whether fraud USD is more or less concentrated
# than ordinary commerce.
print("\n         fraud   legit")
for fraction in (0.01, 0.05, 0.10):
    print(
        f"top {fraction:>4.0%}  {top_share(fraud_amounts, fraction):>6.2%}  "
        f"{top_share(legit_amounts, fraction):>6.2%}"
    )

# The README's headline unit. Gross exposure, nothing intercepting it - every later
# number gets measured against this one.
usd_per_1000 = total_fraud_amount / len(all_amounts) * 1000
print(f"\nGross exposure: {usd_per_1000:,.2f} USD per 1,000 transactions")
print(
    f"                {total_fraud_amount / days:,.0f} USD/day over {transactions_per_day:,.0f} transactions/day"
)

In [ ]:
# Log-spaced bins with a log x-axis, so ticks read in dollars rather than powers of
# ten. Shared bins across both classes, or the shapes are not comparable.
bins = np.logspace(np.log10(all_amounts.min()), np.log10(all_amounts.max()), 60)

fig, ax = plt.subplots(figsize=(11, 5))

# Weighted to each class's own size rather than density=True: these bins are ~1000x
# wider at the right end, and density divides by bin width, so on a log axis that
# draws them equally wide the bars would mislead. Each bar here is "share of class".
for amounts, label in ((legit_amounts, "legitimate"), (fraud_amounts, "fraud")):
    ax.hist(
        amounts,
        bins=bins,
        weights=np.ones(len(amounts)) / len(amounts),
        histtype="step",
        lw=2,
        label=label,
    )

ax.set_xscale("log")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set_xlabel("transaction amount (USD, log scale)")
ax.set_ylabel("share of class")
ax.set_title(f"Amount distribution by class (days 1-{EDA_MAX_DAY})")
ax.legend(frameon=False)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(ROOT / "reports/figures/amount_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Log-spaced bins smooth over spikes at round values, so the histogram above cannot
# answer this. Compare the share each exact amount holds within its own class: the
# ratio column is how over- or under-represented an amount is among frauds.
top_fraud_amounts = fraud_amounts.value_counts(normalize=True).head(15)

round_numbers = pd.DataFrame(
    {
        "fraud_share": top_fraud_amounts,
        "legit_share": legit_amounts.value_counts(normalize=True).reindex(top_fraud_amounts.index),
    }
)
round_numbers["ratio"] = round_numbers["fraud_share"] / round_numbers["legit_share"]
round_numbers.index.name = "amount"
round_numbers

In [ ]:
# Concentration only matters through what it buys. If catches were amount-blind, money
# recovered would track recall exactly; ranking by amount is the counterfactual.
fraud_by_size = fraud_amounts.sort_values(ascending=False).to_numpy()

recovery = pd.DataFrame(
    {
        "ranked_by_amount": {
            r: fraud_by_size[: int(r * len(fraud_by_size))].sum() / total_fraud_amount
            for r in (0.05, 0.10, 0.20, 0.30, 0.50)
        }
    }
)
recovery.index.name = "recall"
recovery["amount_blind"] = recovery.index
recovery["gain"] = recovery["ranked_by_amount"] / recovery["amount_blind"]

# The operating point: review capacity caps recall well below where the two converge.
print(f"review recall ceiling: {reviews_per_day / frauds_per_day:.0%}\n")
recovery

In [ ]:
# The top-15 table above is selected on amounts already common among frauds, which
# mechanically favours high ratios. Scanning every round value regardless of fraud rank
# removes that - and is the only way to see the values fraudsters AVOID.
def exact_ratio(value):
    """Over-representation of an exact amount among frauds, with the fraud count."""
    fraud_share = (fraud_amounts == value).mean()
    legit_share = (legit_amounts == value).mean()
    return (fraud_share / legit_share if legit_share else np.nan), int(
        (fraud_amounts == value).sum()
    )


round_scan = (
    pd.DataFrame(
        [
            {"amount": v, "n_fraud": n, "ratio": r}
            for v in list(range(100, 1101, 100)) + list(range(150, 951, 100))
            for r, n in [exact_ratio(float(v))]
        ]
    )
    .set_index("amount")
    .sort_index()
)

print(f"exact $100 is {(all_amounts == 100).mean():.2%} of all transactions\n")
round_scan

In [ ]:
# Why the histogram cannot answer the round-number question: its tall bin near $100 is
# wide in dollars and holds many distinct amounts, only one of which is round.
tall = int(np.digitize(100.0, bins)) - 1
low, high = bins[tall], bins[tall + 1]
inside = all_amounts[(all_amounts >= low) & (all_amounts < high)]

print(f"the tall bin spans ${low:,.2f} - ${high:,.2f} (width ${high - low:,.2f})")
print(f"it holds {len(inside):,} transactions across {inside.nunique():,} distinct amounts")
print("\nits largest contributors:")
print(inside.value_counts().head(4).to_string())

#### Amounts are compressed, not shifted

Fraud sits **above** legitimate amounts from roughly the 30th to the 98th percentile and
**below** it outside that band in both directions:

| percentile | legit | fraud |
|---|---:|---:|
| 25% | 43.95 | 34.92 |
| 50% | 68.95 | **76.02** |
| 75% | 125.00 | **171.00** |
| 97.5% | 640.95 | **744.95** |
| 99% | 1104.00 | 994.00 |

Fraud is also *less* variable (std 217.5 vs 239.1) despite a higher mean ($146.15 vs
$134.28, +8.8%, t = 6.5) and median (+10.3%).

> **Corrected in section 5.** This pooled reading is a `ProductCD` mix artifact. *Within*
> every product, fraud sits at or above legitimate amounts at every percentile including
> p10 and p25 — the low-end reversal above comes from fraud concentrating in product C
> (12.1% of rows, 38.5% of frauds, median $32.02) while product W dominates the row count
> (72.1%, median $80.00). Simpson's paradox. **H1** in `docs/hypotheses.md` was rewritten
> accordingly.
>
> Worth noting the pooled result was statistically unambiguous (t = 6.5) and structurally
> wrong. Significance does not protect against a mix artifact.

The difference is economically modest either way. Amount alone is a weak separator.

#### The money is spread, not concentrated

Fraud takes **3.821% of USD** against **3.522% of transactions** — a ratio of 1.085,
matching the gap between the two class means. Amount carries a little information about
exposure, but only a little.

Concentration, against legitimate activity as the baseline:

| top | fraud | legit |
|---|---:|---:|
| 1% | 10.25% | 13.75% |
| 5% | 29.27% | 32.64% |
| 10% | 43.28% | 45.12% |

Fraud USD is **less** concentrated than legitimate USD at every cut. The baseline is what
makes that readable: fraud is tail-heavy, but no more so than ordinary commerce. There is
no fraud-specific population of whales to special-case.

That is *not* the same as concentration being unimportant — it matters a great deal:

| recall | fraud USD recovered, ranked by amount | amount-blind | gain |
|---:|---:|---:|---:|
| 5% | 29.3% | 5% | 5.9× |
| 10% | 43.3% | 10% | 4.3× |
| 20% | 61.5% | 20% | 3.1× |
| 30% | 73.2% | 30% | 2.4× |
| 50% | 87.3% | 50% | 1.7× |

The gain is largest at low recall, which is exactly where this system lives: review
capacity of ~35 slots/day against ~122 frauds/day caps review recall at **28%**.

**So unweighted recall is not a safe proxy for dollars saved.** PR-AUC and
recall@capacity can improve while USD saved does not, and the reverse — both have to be
reported. This is also independent justification for a per-transaction threshold
`p* = C_fp / (amount + fee)`: an amount-blind threshold leaves most of the recoverable
money uncollected.

**Gross exposure: 5,147.45 USD per 1,000 transactions** (~17,782 USD/day). Nothing
intercepting it; every later number is measured against this.

#### Round amounts, but only in a band

Round $50 multiples between $150 and $500 are over-represented among frauds — $300 at
4.74×, $450 at 12.94×, $150 at 2.08×. But **$100 runs the other way at 0.68×** despite
being 4.01% of all transactions, and everything above $500 sits at parity. The general
"fraudsters like round numbers" claim does not hold; the banded version does. Recorded
as **H2**.

> **Methodological note.** The histogram cannot show this. Its tall spike near $100 is a
> $21.80-wide bin holding 65,813 transactions across 651 distinct amounts, whose largest
> contributors are $117.00, $100.00 and $107.95. Reading that spike as round-number
> behaviour would invert the actual finding. Binning destroys exactly the information
> the question is about — the `value_counts` comparison is the evidence, not the chart.

### Section 5 - missingness by column family

This data is missing-heavy by design and the missingness is not noise: LightGBM splits on
it, and `load.py` deliberately fills nothing. The question is which absences carry
information and which are artifacts of how the table was assembled.

Three things are separated here. What each family's null rate actually is (means hide a
lot). How much of the identity block's emptiness is the LEFT join rather than sparse
data. And whether absence itself predicts fraud, independently of any value.

In [ ]:
# One 656 MB read, reused by sections 6 and 7 - section 7 needs the same 339 V columns
# and pulling them twice is the expensive mistake.
full_df = read_eda(None)

FAMILIES = {
    "V": r"^V\d+$",
    "C": r"^C\d+$",
    "D": r"^D\d+$",
    "M": r"^M\d$",
    "id": r"^id_\d+$",
    "card": r"^card\d$",
    "addr": r"^addr\d$",
    "dist": r"^dist\d$",
}

# Keys, target, derived time columns and the standalone categoricals belong to no family.
UNFAMILIED = {
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "has_identity",
    "day",
    "hour",
    "weekday",
}

family_columns = {
    name: [c for c in full_df.columns if re.match(pattern, c)] for name, pattern in FAMILIES.items()
}

# Totality check, in the spirit of build_dtype_map: a typo in one pattern would silently
# drop a family, and V is 339 columns to lose quietly.
assigned = set().union(*family_columns.values())
unaccounted = set(full_df.columns) - assigned - UNFAMILIED
assert not unaccounted, f"columns in no family and not exempt: {sorted(unaccounted)}"

null_rate = full_df.isna().mean()

family_nulls = pd.DataFrame(
    {
        "columns": {n: len(c) for n, c in family_columns.items()},
        "mean": {n: null_rate[c].mean() for n, c in family_columns.items()},
        "min": {n: null_rate[c].min() for n, c in family_columns.items()},
        "max": {n: null_rate[c].max() for n, c in family_columns.items()},
    }
)
family_nulls["spread"] = family_nulls["max"] - family_nulls["min"]
family_nulls.index.name = "family"
family_nulls.sort_values("mean", ascending=False)

In [ ]:
# Columns sourced from the identity file are null for every unmatched transaction, so
# their headline null rate mostly measures the join. Conditioning on has_identity is what
# separates "no record at all" from "a record whose field was empty" - the same
# distinction join_identity built the flag for.
identity_sourced = family_columns["id"] + ["DeviceType", "DeviceInfo"]
matched = full_df.loc[full_df["has_identity"]]

identity_nulls = pd.DataFrame(
    {
        "all_rows": null_rate[identity_sourced],
        "matched_only": matched[identity_sourced].isna().mean(),
    }
)
identity_nulls["join_artifact"] = identity_nulls["all_rows"] - identity_nulls["matched_only"]

print(f"rows with no identity record: {(~full_df['has_identity']).mean():.1%}")
print(
    f"id_* mean null rate - all rows {null_rate[family_columns['id']].mean():.1%}, "
    f"matched only {matched[family_columns['id']].isna().mean().mean():.1%}"
)
identity_nulls.sort_values("matched_only", ascending=False).head(12)

In [ ]:
# Does absence itself predict fraud? Compare the fraud rate on rows where an entire
# family is null against the rest. Lift is relative to the other rows, not to the base
# rate, so a family that is almost never null cannot manufacture a large number.
signal = {}
for name, cols in family_columns.items():
    all_null = full_df[cols].isna().all(axis=1)
    if all_null.any() and not all_null.all():
        signal[name] = {
            "share_of_rows": all_null.mean(),
            "fraud_when_null": full_df.loc[all_null, "isFraud"].mean(),
            "fraud_otherwise": full_df.loc[~all_null, "isFraud"].mean(),
        }

missing_signal = pd.DataFrame(signal).T
missing_signal["lift"] = missing_signal["fraud_when_null"] / missing_signal["fraud_otherwise"]
missing_signal.index.name = "family"

# has_identity is the flag join_identity created on the expectation that the absence of a
# record is itself predictive. This is that expectation being tested.
print(full_df.groupby("has_identity")["isFraud"].agg(rate="mean", rows="size").to_string())
print(f"\nbase rate: {full_df['isFraud'].mean():.3%}\n")
missing_signal.sort_values("lift", ascending=False)

#### Controlling for `ProductCD`

The table above makes a missing address look like the strongest signal in the notebook.
Before believing that, it has to survive the one variable that could produce it with no
attacker behaviour involved: what is being bought.

The same control is applied to section 4's amount findings, because a mix artifact in one
place is a reason to suspect the others.

In [ ]:
# What each product looks like, and how much of the address signal it could account for.
addr_null = full_df[family_columns["addr"]].isna().all(axis=1)
by_product = full_df.groupby("ProductCD", observed=True)

product_profile = pd.DataFrame(
    {
        "rows": by_product.size(),
        "share_of_rows": by_product.size() / len(full_df),
        "addr_null_rate": addr_null.groupby(full_df["ProductCD"], observed=True).mean(),
        "has_identity_rate": by_product["has_identity"].mean(),
        "fraud_rate": by_product["isFraud"].mean(),
        "share_of_frauds": by_product["isFraud"].sum() / full_df["isFraud"].sum(),
        "share_of_fraud_usd": by_product.apply(
            lambda g: g.loc[g["isFraud"] == 1, "TransactionAmt"].sum(), include_groups=False
        )
        / full_df.loc[full_df["isFraud"] == 1, "TransactionAmt"].sum(),
        "median_amt": by_product["TransactionAmt"].median(),
    }
).sort_values("fraud_rate", ascending=False)
product_profile

In [ ]:
# Does a null address still lift fraud with product held fixed? If the lift collapses,
# "missing address" was a restatement of "product C".
within = {}
for product, group in full_df.groupby("ProductCD", observed=True):
    is_null = addr_null.loc[group.index]
    if is_null.nunique() < 2:
        continue
    within[product] = {
        "n_null": int(is_null.sum()),
        "fraud_when_null": group.loc[is_null, "isFraud"].mean(),
        "fraud_when_present": group.loc[~is_null, "isFraud"].mean(),
    }

addr_within_product = pd.DataFrame(within).T
addr_within_product["lift"] = (
    addr_within_product["fraud_when_null"] / addr_within_product["fraud_when_present"]
)
addr_within_product.index.name = "ProductCD"

pooled_lift = full_df.loc[addr_null, "isFraud"].mean() / full_df.loc[~addr_null, "isFraud"].mean()
print(f"pooled lift: {pooled_lift:.2f}x")
addr_within_product

In [ ]:
# The same control on has_identity. Section 5 called its direction counterintuitive;
# check whether it is a risk signal at all, or a property of the sales channel.
agreement = (full_df["has_identity"] == (full_df["ProductCD"] != "W")).mean()
w_with_identity = int(full_df.loc[full_df["ProductCD"] == "W", "has_identity"].sum())

print(f"has_identity agrees with (ProductCD != 'W') on {agreement:.2%} of rows")
print(f"product W transactions carrying an identity record: {w_with_identity:,}")

# Where a product still contains both kinds of row, how much lift survives?
residual = {}
for product, group in full_df.groupby("ProductCD", observed=True):
    if group["has_identity"].nunique() < 2:
        continue
    residual[product] = {
        "n_with_identity": int(group["has_identity"].sum()),
        "fraud_with": group.loc[group["has_identity"], "isFraud"].mean(),
        "fraud_without": group.loc[~group["has_identity"], "isFraud"].mean(),
    }

identity_residual = pd.DataFrame(residual).T
identity_residual["lift"] = identity_residual["fraud_with"] / identity_residual["fraud_without"]
identity_residual.index.name = "ProductCD"

pooled_identity_lift = (
    full_df.loc[full_df["has_identity"], "isFraud"].mean()
    / full_df.loc[~full_df["has_identity"], "isFraud"].mean()
)
print(f"pooled lift: {pooled_identity_lift:.2f}x")
identity_residual

In [ ]:
# Section 4 read the pooled percentiles as a compression - fraud below legitimate at the
# low end, above in the middle. Same control applied: fraud amount minus legit amount, in
# USD, at each percentile, within each product.
gap_qs = [0.10, 0.25, 0.50, 0.75, 0.90, 0.99]

gaps = {}
for product, group in full_df.groupby("ProductCD", observed=True):
    fraud = group.loc[group["isFraud"] == 1, "TransactionAmt"]
    legit = group.loc[group["isFraud"] == 0, "TransactionAmt"]
    if len(fraud) < 200:
        continue
    gaps[f"{product} (n={len(fraud):,})"] = (
        fraud.quantile(gap_qs).to_numpy() - legit.quantile(gap_qs).to_numpy()
    )

amount_gap_within_product = pd.DataFrame(gaps, index=[f"p{int(q * 100)}" for q in gap_qs]).round(1)
amount_gap_within_product

In [ ]:
# And the same control on the round-value effect. Ratio is fraud share / legit share at
# that exact amount; blank where a product has fewer than 15 such frauds to judge on.
ROUND_VALUES = [150.0, 200.0, 250.0, 300.0, 450.0, 500.0]


def round_ratio(frame, value):
    """Over-representation of an exact amount among frauds, and the fraud count."""
    fraud = frame.loc[frame["isFraud"] == 1, "TransactionAmt"]
    legit = frame.loc[frame["isFraud"] == 0, "TransactionAmt"]
    if len(fraud) == 0 or not (legit == value).any():
        return np.nan, int((fraud == value).sum())
    return (fraud == value).mean() / (legit == value).mean(), int((fraud == value).sum())


records = []
for value in ROUND_VALUES:
    overall, n_fraud = round_ratio(full_df, value)
    record = {"amount": value, "n_fraud": n_fraud, "overall": overall}
    for product, group in full_df.groupby("ProductCD", observed=True):
        ratio, n = round_ratio(group, value)
        record[product] = ratio if n >= 15 else np.nan
    records.append(record)

round_within_product = pd.DataFrame(records).set_index("amount")

# Which products the round amounts belong to at all - H2 and H3 turn out to concern
# different populations.
print("\ntransactions at those round amounts, by product:")
print(
    full_df.loc[full_df["TransactionAmt"].isin(ROUND_VALUES)]
    .groupby("ProductCD", observed=True)
    .size()
    .to_string()
)

round_within_product

#### Missingness is structural, and in two places it is signal

**Family means are misleading for V and D.** V spans 0.0%–84.3% null and D spans
0.1%–93.5%; quoting "V is 42.7% missing" describes no column in it. Both families are
clearly several blocks with different fill behaviour — which is what section 7 goes
looking for. `C1`–`C14` are the opposite: **0.0% null across all fourteen**, consistent
with vendor-computed aggregates that always have a value.

**Most of the identity block's emptiness is the join, not sparse data.** The `id_*` mean
null rate falls from **83.4% to 37.6%** once restricted to rows that matched an identity
record — 73.4% of transactions have no record at all. But conditioning does not rescue
everything: `id_07`, `id_08`, `id_21`–`id_27` stay ~96% null *even when a record exists*,
so those are genuinely sparse fields. `id_03`, `id_04` and `id_18` are the reverse, where
most of the apparent emptiness was the join. Only the conditional number distinguishes
them, which is what `has_identity` was built for.

#### Two families where absence predicts fraud

| family | share of rows all-null | fraud when null | otherwise | lift |
|---|---:|---:|---:|---:|
| addr | 11.5% | **11.28%** | 2.51% | **4.48×** |
| M | 17.1% | 4.23% | 3.38% | 1.25× |
| dist | 54.8% | 3.82% | 3.16% | 1.21× |
| id | 73.4% | 2.14% | 7.34% | 0.29× |

*(V, C, D and card never go entirely null, so there is no contrast to measure.)*

A missing billing address looks like the strongest single signal here — 11.28% fraud
against 2.51%, on 11.5% of rows. `addr1` and `addr2` have identical null rates (spread
0.0000), so they vanish together: one signal, not two.

> **It does not survive `ProductCD`.** 94.2% of product C rows have no address, and C is
> the high-fraud product at 11.20%. Conditioning on product, the residual lift within C is
> **1.30×** (11.36% vs 8.70%, n=47,241) — real but far smaller. Every other product has
> only 47–153 null-address rows, too few to judge. "Missing address" is largely a
> restatement of "product C", which is why **H3** was written about `ProductCD` itself.

**`has_identity` runs in the direction opposite to intuition.** Fraud is **7.34%** where
an identity record exists and **2.14%** where it does not — 3.42× higher *with* identity,
not without. The flag is strongly predictive, and `join_identity` was right to build it
explicitly rather than infer it from nulls.

> **The direction is mostly the sales channel, not a risk signal.** Of 298,873 product W
> transactions, **zero** carry an identity record, while 95.2% of non-W transactions do.
> `has_identity` and `ProductCD != "W"` agree on **98.66%** of rows — it is very nearly a
> product indicator wearing another name, and W is both the largest product (72.1%) and
> the lowest-risk one (2.08%).
>
> A residual survives inside each product, where both kinds of row exist: C 2.08×, S 1.57×,
> H 1.48×, R 1.47×. So among transactions where identity is normally collected, the ones
> lacking it really are less risky — but at a fraction of the pooled 3.42×.
>
> That residual is where the provenance worry lives. If collection is occasionally
> triggered by suspicion rather than by channel, the flag encodes *a prior fraud decision*
> on those rows, in the same family as the ROADMAP's warning about `C*`/`D*` vendor
> aggregates. Recorded as **A8** in `problem-statement.md` — a smaller concern than it
> first appeared, and a more specific one.

**Servability.** Both signals are cheap at scoring time — `addr1`/`addr2` presence and
whether an identity payload arrived are properties of the request itself, needing no
entity history. Unlike velocity features, these cost nothing to serve.

### Section 6 - entity concentration, as H3's falsification test

H3 claims `ProductCD` is the strongest single categorical predictor, and names its own way
of being wrong: *another single field separating risk more sharply, with `card` and `addr`
concentration the obvious contenders.* This section runs that test rather than leaving the
criterion decorative.

Cardinality is an input, not the answer. `card1` has thousands of levels and can
manufacture extreme fraud rates on cells of five rows, so any comparison against
`ProductCD`'s five levels has to control for support. Two guards: only levels with at least
1,000 rows count, and separation is measured at **fixed volume coverage** — the share of
fraud sitting inside the riskiest levels covering 10% of transactions, which is the same
shape as the recall@capacity constraint the system actually operates under.

Both guards still leave the comparison biased *toward* the high-cardinality contenders,
because levels are ranked on the same data the capture is measured on. That is deliberate:
if `ProductCD` wins anyway, the conclusion holds a fortiori.

In [ ]:
ENTITY_COLUMNS = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "ProductCD",
]

entity_profile = pd.DataFrame(
    {
        "levels": {c: full_df[c].nunique() for c in ENTITY_COLUMNS},
        "null_rate": {c: full_df[c].isna().mean() for c in ENTITY_COLUMNS},
        # How top-heavy the field is: if ten levels cover most rows, it behaves like a
        # low-cardinality field regardless of how many levels exist in the tail.
        "top10_row_share": {
            c: full_df[c].value_counts(normalize=True).head(10).sum() for c in ENTITY_COLUMNS
        },
    }
).sort_values("levels", ascending=False)
entity_profile

In [ ]:
MIN_SUPPORT = 1_000


def level_stats(column, min_support=MIN_SUPPORT):
    """Per-level rows, frauds and fraud rate, keeping only well-supported levels.

    `dropna=False` because a null is a level here - section 5 showed a missing address is
    not an absence of information.
    """
    stats = full_df.groupby(column, observed=True, dropna=False)["isFraud"].agg(
        rows="size", frauds="sum"
    )
    stats["rate"] = stats["frauds"] / stats["rows"]
    return stats[stats["rows"] >= min_support].sort_values("rate", ascending=False)


separation = {}
for column in ENTITY_COLUMNS:
    stats = level_stats(column)
    if len(stats) < 2:
        continue
    separation[column] = {
        "levels_kept": len(stats),
        "rows_covered": stats["rows"].sum() / len(full_df),
        "min_rate": stats["rate"].min(),
        "max_rate": stats["rate"].max(),
        "spread": stats["rate"].max() / stats["rate"].min() if stats["rate"].min() > 0 else np.inf,
    }

separation_table = pd.DataFrame(separation).T.sort_values("spread", ascending=False)
print(f"levels with at least {MIN_SUPPORT:,} rows only\n")
separation_table

In [ ]:
def capture_at(column, volume_share=0.10, min_support=MIN_SUPPORT):
    """Share of all frauds inside the riskiest levels covering ~`volume_share` of rows.

    The level that crosses the threshold is included rather than dropped: a coarse field
    like ProductCD has a single level covering 12% of rows, and excluding it would score
    the field at zero. `volume_covered` is returned so the comparison stays honest, and
    `lift` normalises capture by the volume actually spent.

    Levels are ranked by fraud rate on the same data the capture is measured on, which
    flatters high-cardinality fields. See the section note.
    """
    stats = level_stats(column, min_support)
    if stats.empty:
        return np.nan, 0.0, 0
    covered = stats["rows"].cumsum() / len(full_df)
    # Coverage *before* each level: include a level if there was still room before it.
    keep = covered.shift(1, fill_value=0.0) < volume_share
    return (
        stats.loc[keep, "frauds"].sum() / full_df["isFraud"].sum(),
        float(covered[keep].max()),
        int(keep.sum()),
    )


capture = {}
for column in ENTITY_COLUMNS:
    fraud_share, volume, levels = capture_at(column)
    capture[column] = {
        "levels_used": levels,
        "volume_covered": volume,
        "fraud_captured": fraud_share,
        "lift": fraud_share / volume if volume else np.nan,
    }

# addr1/addr2 score identically above. Are they a separate signal from ProductCD, or the
# same rows counted twice?
print(
    f"addr1 and addr2 share a null mask: {full_df['addr1'].isna().equals(full_df['addr2'].isna())}"
)
print(f"addr-null rows: {addr_null.sum():,}")
print(f"  of which ProductCD C: {(full_df.loc[addr_null, 'ProductCD'] == 'C').mean():.1%}")

capture_table = pd.DataFrame(capture).T.sort_values("lift", ascending=False)
print("riskiest levels covering ~10% of transactions (boundary level included)\n")
capture_table

#### H3 survives, but only just — and the raw spread was a trap

**Raw fraud-rate spread favours the high-cardinality fields**, across levels with at least
1,000 rows:

| field | levels kept | spread |
|---|---:|---:|
| card2 | 57 | **57.2×** |
| card1 | 73 | 46.3× |
| addr1 | 48 | 21.6× |
| card5 | 15 | 17.0× |
| ProductCD | 5 | 5.4× |

Read alone, that falsifies H3 outright — `card2` separates risk ten times as sharply.

**At fixed volume it reverses.** Ranking each field's levels by fraud rate and taking the
riskiest ones covering ~10% of transactions:

| field | levels used | volume | fraud captured | lift |
|---|---:|---:|---:|---:|
| addr1 / addr2 | 1 | 11.5% | 36.8% | **3.20×** |
| ProductCD | 1 | 12.1% | 38.5% | **3.18×** |
| card2 | 12 | 10.2% | 32.0% | **3.13×** |
| card5 | 4 | 11.2% | 27.1% | 2.43× |
| card1 | 19 | 10.3% | 21.9% | 2.14× |

`card1`, with **12,251 levels**, captures the *least* — 2.14× against `ProductCD`'s 3.18×
from a single level. Its 46× spread came from wide variation across many small cells, not
from separating the bulk of the traffic. That distinction is invisible in a spread column
and decisive operationally.

Note the bias runs the other way from the conclusion: levels were ranked on the same data
the capture is measured on, which *flatters* `card1` and `card2`. They lose anyway.

**The top three are a statistical tie** — 3.20, 3.18, 3.13. So H3's claim that `ProductCD`
is *the strongest* is too strong; it is *among* the strongest.

**And `addr` is not an independent contender.** `addr1` and `addr2` share an identical null
mask, and **99.2% of address-null rows are ProductCD C**. Their 3.20× is `ProductCD`'s
result arriving under a different name — which is the proxying H3 predicted, showing up in
its own falsification test.

`card2` is the one genuinely separate contender, at 3.13× across 12 levels. It is not a
free feature, though: 500 levels means target encoding fitted inside CV folds, and Phase 04
owns that. `ProductCD` costs nothing and arrives with the request.

### Section 7 - V-block correlation structure: deferred to Phase 04

Not an omission. The ROADMAP lists `V*` correlation structure among the Phase 01 profiling
steps, and its own trap note for this phase says: *"Don't drop columns by gut feel now;
decide with evidence in Phase 4."* Computing it here produces evidence that may not be
acted on for three phases.

The stronger reason is that the analysis would be **wrong if done now**. Encoders and
aggregates fit on the training window only, and correlation among 339 columns is exactly
the kind of statistic Phase 04 would use to drop features. That window does not exist yet
— Phase 02 defines it. Measuring correlation across days 1–120 means either recomputing it
later, or being tempted to reuse a number that spans what will become `VAL-FIT` and
`VAL-CAL`.

One piece is safe to compute at any time: grouping V columns by **identical NaN pattern**
is structural rather than fitted, so no window applies. Section 5 already showed the family
needs it — V spans 0.0% to 84.3% null, so the family mean describes no column in it. That
grouping is recorded here as the right first move for Phase 04 rather than run now, since
its only consumer is the feature-selection decision that phase owns.

**What Phase 04 should do with this:** group by NaN pattern first, then measure correlation
*within* the training split only, then decide drops. In that order.